## Used Car Price Prediction

Given *data about used cars*, let's try to predict the **price** of a given car.

We will use linear regression and gradient boosting (LightGBM) to make our predictions.

Data source: https://www.kaggle.com/datasets/austinreese/craigslist-carstrucks-data

### Importing Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
import lightgbm as lgb

from sklearn.metrics import mean_squared_error

pd.set_option('display.max_columns', None)

In [2]:
data = pd.read_csv('archive/vehicles.csv', nrows=100000)
data

,id,url,region,region_url,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,VIN,drive,size,type,paint_color,image_url,description,county,state,lat,long,posting_date
0,7222695916,https://prescott.craigslist.org/cto/d/prescott...,prescott,https://prescott.craigslist.org,6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,az,NaN,NaN,NaN
1,7218891961,https://fayar.craigslist.org/ctd/d/bentonville...,fayetteville,https://fayar.craigslist.org,11900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ar,NaN,NaN,NaN
2,7221797935,https://keys.craigslist.org/cto/d/summerland-k...,florida keys,https://keys.craigslist.org,21000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fl,NaN,NaN,NaN
3,7222270760,https://worcester.craigslist.org/cto/d/west-br...,worcester / central MA,https://worcester.craigslist.org,1500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ma,NaN,NaN,NaN
4,7210384030,https://greensboro.craigslist.org/cto/d/trinit...,greensboro,https://greensboro.craigslist.org,4900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nc,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,7309358353,https://jacksonville.craigslist.org/ctd/d/jack...,jacksonville,https://jacksonville.craigslist.org,49395,2019.0,chevrolet,silverado trail boss,new,8 cylinders,gas,12700.0,clean,automatic,1GCPYCEF1KZ338219,4wd,full-size,pickup,red,https://images.craigslist.org/00x0x_cg9hmWpOhy...,"*** MINT CONDITION *** BRAND NEW 7"" PRO COMP. ...",NaN,fl,30.207288,-81.738969,2021-04-19T13:50:58-0400
99996,7309358251,https://jacksonville.craigslist.org/ctd/d/jack...,jacksonville,https://jacksonville.craigslist.org,18590,2018.0,kia,sportage lx sport utility 4d,good,NaN,other,20005.0,clean,other,KNDPMCACXJ7395729,NaN,NaN,other,silver,https://images.craigslist.org/00o0o_4QCkqtOcFy...,Carvana is the safer way to buy a car During t...,NaN,fl,30.330000,-81.650000,2021-04-19T13:50:50-0400
99997,7309355294,https://jacksonville.craigslist.org/ctd/d/jack...,jacksonville,https://jacksonville.craigslist.org,49495,2019.0,chevrolet,silverado trail boss,new,8 cylinders,gas,12700.0,clean,automatic,1GCPYCEF1KZ338219,4wd,full-size,pickup,red,https://images.craigslist.org/00x0x_cg9hmWpOhy...,"*** MINT CONDITION *** BRAND NEW 7"" PRO COMP. ...",NaN,fl,30.207288,-81.738969,2021-04-19T13:46:36-0400
99998,7309354677,https://jacksonville.craigslist.org/ctd/d/jack...,jacksonville,https://jacksonville.craigslist.org,24495,2014.0,chevrolet,silverado 1500 lt 4x4,excellent,8 cylinders,gas,100166.0,clean,automatic,1GCVKREC7EZ391819,4wd,full-size,truck,white,https://images.craigslist.org/00f0f_djNCWjHFxp...,*** MINT CONDITION *** CLEAN CARFAX - NO ACCID...,NaN,fl,30.207288,-81.738969,2021-04-19T13:45:40-0400


In [3]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 26 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            100000 non-null  int64  
 1   url           100000 non-null  str    
 2   region        100000 non-null  str    
 3   region_url    100000 non-null  str    
 4   price         100000 non-null  int64  
 5   year          99572 non-null   float64
 6   manufacturer  95580 non-null   str    
 7   model         98715 non-null   str    
 8   condition     61711 non-null   str    
 9   cylinders     59841 non-null   str    
 10  fuel          99397 non-null   str    
 11  odometer      98479 non-null   float64
 12  title_status  97721 non-null   str    
 13  transmission  99486 non-null   str    
 14  VIN           63962 non-null   str    
 15  drive         69526 non-null   str    
 16  size          27846 non-null   str    
 17  type          80389 non-null   str    
 18  paint_color   70

### Preprocessing

In [4]:
data.isna().sum()

id                   0
url                  0
region               0
region_url           0
price                0
year               428
manufacturer      4420
model             1285
condition        38289
cylinders        40159
fuel               603
odometer          1521
title_status      2279
transmission       514
VIN              36038
drive            30474
size             72154
type             19611
paint_color      29776
image_url           38
description         39
county          100000
state                0
lat                539
long               539
posting_date        38
dtype: int64

In [5]:
null_columns = data.columns[data.isna().mean() >= 0.25]
null_columns

Index(['condition', 'cylinders', 'VIN', 'drive', 'size', 'paint_color',
       'county'],
      dtype='str')

In [6]:
data = data.drop(null_columns, axis=1)

In [7]:
data

,id,url,region,region_url,price,year,manufacturer,model,fuel,odometer,title_status,transmission,type,image_url,description,state,lat,long,posting_date
0,7222695916,https://prescott.craigslist.org/cto/d/prescott...,prescott,https://prescott.craigslist.org,6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,az,NaN,NaN,NaN
1,7218891961,https://fayar.craigslist.org/ctd/d/bentonville...,fayetteville,https://fayar.craigslist.org,11900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ar,NaN,NaN,NaN
2,7221797935,https://keys.craigslist.org/cto/d/summerland-k...,florida keys,https://keys.craigslist.org,21000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fl,NaN,NaN,NaN
3,7222270760,https://worcester.craigslist.org/cto/d/west-br...,worcester / central MA,https://worcester.craigslist.org,1500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ma,NaN,NaN,NaN
4,7210384030,https://greensboro.craigslist.org/cto/d/trinit...,greensboro,https://greensboro.craigslist.org,4900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nc,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,7309358353,https://jacksonville.craigslist.org/ctd/d/jack...,jacksonville,https://jacksonville.craigslist.org,49395,2019.0,chevrolet,silverado trail boss,gas,12700.0,clean,automatic,pickup,https://images.craigslist.org/00x0x_cg9hmWpOhy...,"*** MINT CONDITION *** BRAND NEW 7"" PRO COMP. ...",fl,30.207288,-81.738969,2021-04-19T13:50:58-0400
99996,7309358251,https://jacksonville.craigslist.org/ctd/d/jack...,jacksonville,https://jacksonville.craigslist.org,18590,2018.0,kia,sportage lx sport utility 4d,other,20005.0,clean,other,other,https://images.craigslist.org/00o0o_4QCkqtOcFy...,Carvana is the safer way to buy a car During t...,fl,30.330000,-81.650000,2021-04-19T13:50:50-0400
99997,7309355294,https://jacksonville.craigslist.org/ctd/d/jack...,jacksonville,https://jacksonville.craigslist.org,49495,2019.0,chevrolet,silverado trail boss,gas,12700.0,clean,automatic,pickup,https://images.craigslist.org/00x0x_cg9hmWpOhy...,"*** MINT CONDITION *** BRAND NEW 7"" PRO COMP. ...",fl,30.207288,-81.738969,2021-04-19T13:46:36-0400
99998,7309354677,https://jacksonville.craigslist.org/ctd/d/jack...,jacksonville,https://jacksonville.craigslist.org,24495,2014.0,chevrolet,silverado 1500 lt 4x4,gas,100166.0,clean,automatic,truck,https://images.craigslist.org/00f0f_djNCWjHFxp...,*** MINT CONDITION *** CLEAN CARFAX - NO ACCID...,fl,30.207288,-81.738969,2021-04-19T13:45:40-0400


In [8]:
unneeded_columns = ['id', 'url', 'region_url', 'image_url', 'description']

data = data.drop(unneeded_columns, axis=1)

In [9]:
data

,region,price,year,manufacturer,model,fuel,odometer,title_status,transmission,type,state,lat,long,posting_date
0,prescott,6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,az,NaN,NaN,NaN
1,fayetteville,11900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ar,NaN,NaN,NaN
2,florida keys,21000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fl,NaN,NaN,NaN
3,worcester / central MA,1500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ma,NaN,NaN,NaN
4,greensboro,4900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nc,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,jacksonville,49395,2019.0,chevrolet,silverado trail boss,gas,12700.0,clean,automatic,pickup,fl,30.207288,-81.738969,2021-04-19T13:50:58-0400
99996,jacksonville,18590,2018.0,kia,sportage lx sport utility 4d,other,20005.0,clean,other,other,fl,30.330000,-81.650000,2021-04-19T13:50:50-0400
99997,jacksonville,49495,2019.0,chevrolet,silverado trail boss,gas,12700.0,clean,automatic,pickup,fl,30.207288,-81.738969,2021-04-19T13:46:36-0400
99998,jacksonville,24495,2014.0,chevrolet,silverado 1500 lt 4x4,gas,100166.0,clean,automatic,truck,fl,30.207288,-81.738969,2021-04-19T13:45:40-0400


In [10]:
{column: len(data[column].unique()) for column in data.columns if data.dtypes[column] == 'str'}

{'region': 84,
 'manufacturer': 42,
 'model': 12616,
 'fuel': 6,
 'title_status': 7,
 'transmission': 4,
 'type': 14,
 'state': 18,
 'posting_date': 94278}

In [11]:
data = data.drop('model', axis=1)

In [12]:
data['posting_date'] = pd.to_datetime(data['posting_date'], utc=True)

In [13]:
data['posting_year'] = data['posting_date'].dt.year
data['posting_month'] = data['posting_date'].dt.month
data['posting_day'] = data['posting_date'].dt.day

data = data.drop('posting_date', axis=1)

In [14]:
data

,region,price,year,manufacturer,fuel,odometer,title_status,transmission,type,state,lat,long,posting_year,posting_month,posting_day
0,prescott,6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,az,NaN,NaN,NaN,NaN,NaN
1,fayetteville,11900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ar,NaN,NaN,NaN,NaN,NaN
2,florida keys,21000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fl,NaN,NaN,NaN,NaN,NaN
3,worcester / central MA,1500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ma,NaN,NaN,NaN,NaN,NaN
4,greensboro,4900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nc,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,jacksonville,49395,2019.0,chevrolet,gas,12700.0,clean,automatic,pickup,fl,30.207288,-81.738969,2021.0,4.0,19.0
99996,jacksonville,18590,2018.0,kia,other,20005.0,clean,other,other,fl,30.330000,-81.650000,2021.0,4.0,19.0
99997,jacksonville,49495,2019.0,chevrolet,gas,12700.0,clean,automatic,pickup,fl,30.207288,-81.738969,2021.0,4.0,19.0
99998,jacksonville,24495,2014.0,chevrolet,gas,100166.0,clean,automatic,truck,fl,30.207288,-81.738969,2021.0,4.0,19.0


In [15]:
def onehot_encode(df, columns, prefixes):
    df = df.copy()
    for column, prefix in zip(columns, prefixes):
        dummies = pd.get_dummies(df[column], prefix=prefix)
        df = pd.concat([df, dummies], axis=1)
        df = df.drop(column, axis=1)
    return df

In [16]:
{column: len(data[column].unique()) for column in data.columns if data.dtypes[column] == 'str'}

{'region': 84,
 'manufacturer': 42,
 'fuel': 6,
 'title_status': 7,
 'transmission': 4,
 'type': 14,
 'state': 18}

In [17]:
data = onehot_encode(
    data,
    ['region', 'manufacturer', 'fuel', 'title_status', 'transmission', 'type', 'state'],
    ['RE', 'MA', 'FU', 'TS', 'TR', 'TY', 'ST']
)

In [18]:
data

,price,year,odometer,lat,long,posting_year,posting_month,posting_day,RE_SF bay area,RE_anchorage / mat-su,RE_auburn,RE_bakersfield,RE_bellingham,RE_birmingham,RE_boulder,RE_chico,RE_colorado springs,RE_daytona beach,RE_delaware,RE_denver,RE_dothan,RE_eastern CO,RE_eastern CT,RE_el paso,RE_erie,RE_fairbanks,RE_fayetteville,RE_flagstaff / sedona,RE_florence / muscle shoals,RE_florida keys,RE_fort collins / north CO,RE_fort smith,RE_fresno / madera,RE_ft myers / SW florida,RE_gadsden-anniston,RE_gainesville,RE_gold country,RE_greensboro,RE_hanford-corcoran,RE_hartford,RE_heartland florida,RE_high rockies,RE_hudson valley,RE_humboldt county,RE_huntsville / decatur,RE_imperial county,RE_inland empire,RE_jacksonville,RE_jonesboro,RE_kenai peninsula,RE_la crosse,RE_little rock,RE_los angeles,RE_medford-ashland,RE_mendocino county,RE_merced,RE_mobile,RE_modesto,RE_mohave county,RE_monterey bay,RE_montgomery,RE_new haven,RE_northwest CT,RE_orange county,RE_palm springs,RE_phoenix,RE_prescott,RE_pueblo,RE_redding,RE_reno / tahoe,RE_sacramento,RE_san diego,RE_san luis obispo,RE_santa barbara,RE_santa maria,RE_show low,RE_sierra vista,RE_siskiyou county,RE_skagit / island / SJI,RE_southeast alaska,RE_stockton,RE_susanville,RE_texarkana,RE_tucson,RE_tuscaloosa,RE_ventura county,RE_visalia-tulare,"RE_washington, DC",RE_western slope,RE_worcester / central MA,RE_yuba-sutter,RE_yuma,MA_acura,MA_alfa-romeo,MA_aston-martin,MA_audi,MA_bmw,MA_buick,MA_cadillac,MA_chevrolet,MA_chrysler,MA_datsun,MA_dodge,MA_ferrari,MA_fiat,MA_ford,MA_gmc,MA_harley-davidson,MA_honda,MA_hyundai,MA_infiniti,MA_jaguar,MA_jeep,MA_kia,MA_land rover,MA_lexus,MA_lincoln,MA_mazda,MA_mercedes-benz,MA_mercury,MA_mini,MA_mitsubishi,MA_nissan,MA_pontiac,MA_porsche,MA_ram,MA_rover,MA_saturn,MA_subaru,MA_tesla,MA_toyota,MA_volkswagen,MA_volvo,FU_diesel,FU_electric,FU_gas,FU_hybrid,FU_other,TS_clean,TS_lien,TS_missing,TS_parts only,TS_rebuilt,TS_salvage,TR_automatic,TR_manual,TR_other,TY_SUV,TY_bus,TY_convertible,TY_coupe,TY_hatchback,TY_mini-van,TY_offroad,TY_other,TY_pickup,TY_sedan,TY_truck,TY_van,TY_wagon,ST_ak,ST_al,ST_ar,ST_az,ST_ca,ST_co,ST_ct,ST_dc,ST_de,ST_fl,ST_ma,ST_nc,ST_ny,ST_or,ST_pa,ST_tx,ST_wa,ST_wi
0,6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,11900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,Fal

In [19]:
for column in data.columns:
    data[column] = data[column].fillna(data[column].mean())

In [21]:
data.isna().sum().sum()

np.int64(0)

#### Splitting and Scaling

In [22]:
y = data.loc[:, 'price'].copy()
X = data.drop('price', axis=1).copy()

In [23]:
scaler = StandardScaler()

X = scaler.fit_transform(X)

In [24]:
X

array([[ 2.26586964e-14,  0.00000000e+00,  0.00000000e+00, ...,
        -7.07124460e-03, -1.04886654e-02, -3.16229347e-03],
       [ 2.26586964e-14,  0.00000000e+00,  0.00000000e+00, ...,
        -7.07124460e-03, -1.04886654e-02, -3.16229347e-03],
       [ 2.26586964e-14,  0.00000000e+00,  0.00000000e+00, ...,
        -7.07124460e-03, -1.04886654e-02, -3.16229347e-03],
       ...,
       [ 8.02786577e-01, -3.63243599e-01, -1.10545551e+00, ...,
        -7.07124460e-03, -1.04886654e-02, -3.16229347e-03],
       [ 3.04516573e-01,  1.61174040e-02, -1.10545551e+00, ...,
        -7.07124460e-03, -1.04886654e-02, -3.16229347e-03],
       [-2.18683345e+00,  6.31070584e-02, -1.35637874e+00, ...,
        -7.07124460e-03, -1.04886654e-02, -3.16229347e-03]],
      shape=(100000, 177))

In [26]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, shuffle=True, random_state=34)

### Training

In [27]:
lin_model = LinearRegression()

lin_model.fit(X_train, y_train)

lin_y_preds = lin_model.predict(X_test)

In [28]:
lgb_model = lgb.LGBMRegressor(
    boosting_type = 'gbdt',
    num_leaves = 31,
    n_estimators = 100,
    reg_lambda = 1.0
)

lgb_model.fit(X_train, y_train)

lgb_y_preds = lgb_model.predict(X_test)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.078311 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1316
[LightGBM] [Info] Number of data points in the train set: 70000, number of used features: 155
[LightGBM] [Info] Start training from score 33229.276900


In [29]:
lin_loss = np.sqrt(mean_squared_error(y_test, lin_y_preds))
lgb_loss = np.sqrt(mean_squared_error(y_test, lgb_y_preds))

In [32]:
print("Linear Regression RMSE:", lin_loss)
print("Gradient Boosted RMSE: ", lgb_loss)

Linear Regression RMSE: 26253353.8109846
Gradient Boosted RMSE:  26258360.60125621


In [34]:
print("Linear Regression R^2 Score:", lin_model.score(X_test, y_test))
print(" Gradient Boosted R^2 Score:", lgb_model.score(X_test, y_test))

Linear Regression R^2 Score: -0.0001410147787073157
 Gradient Boosted R^2 Score: -0.00052252594334945
